# NV-Embed-v2 Embedding Stage

SID-v2 movie text embeddings for later RQ-VAE. Run the heavy model cells yourself.

- meta_text = title + year + genres
- description_text = overview
- output prefix: nv_embed_v2_audited_v1


## 0. Environment


In [1]:
%pip install -q datasets einops hf_xet "pyarrow>=15"


Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import json
import os
import re
import time

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import pyarrow
import datasets
import einops
import torch

ROOT = Path(r"C:/Users/User/plum-ml1m-repro")

SOURCE_CSV = Path(r"D:/user/Downloads/ml1m_movie_overviews_sample720_r9_checked_v2_audited_next200_r7_final_audit100.csv")
MOVIES_REINDEXED_PATH = ROOT / "data/processed/movies_reindexed.parquet"
OUT_DIR = ROOT / "data/processed/item_features"

RUN_NAME = "nv_embed_v2_audited_v1"
PROFILE_PATH = OUT_DIR / f"{RUN_NAME}_item_profiles.parquet"
EMBEDDING_PATH = OUT_DIR / f"{RUN_NAME}_meta_desc_embeddings.npz"
MANIFEST_PATH = OUT_DIR / f"{RUN_NAME}_embedding_manifest.json"

MODEL_NAME = "nvidia/NV-Embed-v2"
MAX_SEQ_LENGTH = 1024
TRUNCATE_DIM = None
META_BATCH_SIZE = 1
DESC_BATCH_SIZE = 1
NORMALIZE_EMBEDDINGS = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

OUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE


'cuda'

## 1. Load data

Use project `item_idx`; use audited CSV fields for movie text.

In [3]:
audited = pd.read_csv(SOURCE_CSV)
movies_reindexed = pd.read_parquet(MOVIES_REINDEXED_PATH, columns=["movie_id", "item_idx"])

base_cols = ["movie_id", "title", "year", "genres", "overview", "source", "status"]
audited_core = audited[base_cols].copy()

profiles = (
    movies_reindexed[["movie_id", "item_idx"]]
    .merge(audited_core, on="movie_id", how="left", validate="one_to_one")
    .sort_values("item_idx")
    .reset_index(drop=True)
)

assert profiles["overview"].notna().all()
assert np.array_equal(profiles["item_idx"].to_numpy(), np.arange(len(profiles)))

profiles.head()

,movie_id,item_idx,title,year,genres,overview,source,status
0,1,0,Toy Story (1995),1995,Animation|Children's|Comedy,"Led by Woody, Andy's toys live happily in his ...",https://huggingface.co/datasets/mt0rm0/movie_d...,found
1,2,1,Jumanji (1995),1995,Adventure|Children's|Fantasy,When siblings Judy and Peter discover an encha...,https://huggingface.co/datasets/mt0rm0/movie_d...,found
2,3,2,Grumpier Old Men (1995),1995,Comedy|Romance,A family wedding reignites the ancient feud be...,https://huggingface.co/datasets/mt0rm0/movie_d...,found
3,4,3,Waiting to Exhale (1995),1995,Comedy|Drama,"Cheated on, mistreated and stepped on, the wom...",https://huggingface.co/datasets/mt0rm0/movie_d...,found
4,5,4,Father of the Bride Part II (1995),1995,Comedy,Just when George Banks has recovered from his ...,https://huggingface.co/datasets/mt0rm0/movie_d...,found


## 2. Build text fields

Keep two separate texts: one for metadata, one for plot overview.

In [4]:
YEAR_RE = re.compile(r"\s*\((\d{4})\)\s*$")

def clean_title(x):
    return YEAR_RE.sub("", str(x).strip()).strip()

def clean_year(x):
    return "unknown" if pd.isna(x) else str(int(float(x)))

def clean_genres(x):
    return ", ".join(part.strip() for part in str(x).split("|") if part.strip())

def clean_text(x):
    return re.sub(r"\s+", " ", str(x).replace("\n", " ").replace("\r", " ")).strip()

profiles["title_clean"] = profiles["title"].map(clean_title)
profiles["year_text"] = profiles["year"].map(clean_year)
profiles["genres_text"] = profiles["genres"].map(clean_genres)
profiles["overview_clean"] = profiles["overview"].map(clean_text)

profiles["meta_text"] = profiles.apply(
    lambda r: f"Movie title: {r.title_clean}. Release year: {r.year_text}. Genres: {r.genres_text}.",
    axis=1,
)
profiles["description_text"] = profiles["overview_clean"].map(lambda x: f"Plot overview: {x}")

profiles.to_parquet(PROFILE_PATH, index=False)
profiles[["item_idx", "title", "meta_text", "description_text"]].head()

,item_idx,title,meta_text,description_text
0,0,Toy Story (1995),Movie title: Toy Story. Release year: 1995. Ge...,"Plot overview: Led by Woody, Andy's toys live ..."
1,1,Jumanji (1995),Movie title: Jumanji. Release year: 1995. Genr...,Plot overview: When siblings Judy and Peter di...
2,2,Grumpier Old Men (1995),Movie title: Grumpier Old Men. Release year: 1...,Plot overview: A family wedding reignites the ...
3,3,Waiting to Exhale (1995),Movie title: Waiting to Exhale. Release year: ...,"Plot overview: Cheated on, mistreated and step..."
4,4,Father of the Bride Part II (1995),Movie title: Father of the Bride Part II. Rele...,Plot overview: Just when George Banks has reco...


## 3. Model config smoke test


In [5]:
from transformers import AutoConfig

cfg = AutoConfig.from_pretrained(MODEL_NAME, trust_remote_code=True)
{
    "config_class": cfg.__class__.__name__,
    "hidden_size": int(cfg.hidden_size),
    "model_type": cfg.model_type,
}


C:\Users\User\.conda\conda.new\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


{'config_class': 'NVEmbedConfig', 'hidden_size': 4096, 'model_type': 'nvembed'}

## 4. Download model files


In [6]:
from huggingface_hub import snapshot_download

MODEL_DIR = snapshot_download(
    repo_id=MODEL_NAME,
    allow_patterns=[
        "config.json",
        "configuration_nvembed.py",
        "modeling_nvembed.py",
        "model.safetensors.index.json",
        "model-*.safetensors",
    ],
    max_workers=1,
)
MODEL_DIR


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/789M [00:00<?, ?B/s]

'C:\\Users\\User\\.cache\\huggingface\\hub\\models--nvidia--NV-Embed-v2\\snapshots\\3fa59658547db50a1e8e3346cf057fd0c77ed6ef'

## 5. Check downloaded shards


In [7]:
model_dir = Path(MODEL_DIR)
[(p.name, round(p.stat().st_size / 1024**3, 2)) for p in sorted(model_dir.glob("model-*.safetensors"))]


[('model-00001-of-00004.safetensors', 4.65),
 ('model-00002-of-00004.safetensors', 4.58),
 ('model-00003-of-00004.safetensors', 4.66),
 ('model-00004-of-00004.safetensors', 0.73)]

## 6. Load NV-Embed-v2


In [8]:
from transformers import AutoModel
import torch.nn.functional as F

torch.cuda.empty_cache()
model_kwargs = {"torch_dtype": torch.float16} if DEVICE == "cuda" else {}

model = AutoModel.from_pretrained(
    MODEL_DIR,
    trust_remote_code=True,
    local_files_only=True,
    **model_kwargs,
)
model = model.to(DEVICE)
model.eval()
model


tokenizer_config.json:   0%|          | 0.00/997 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

NVEmbedModel(
  (latent_attention_model): LatentAttentionModel(
    (cross_attend_blocks): ModuleList(
      (0): PreNorm(
        (fn): Attention(
          (to_q): Linear(in_features=4096, out_features=32768, bias=False)
          (to_kv): Linear(in_features=4096, out_features=65536, bias=False)
          (to_out): Linear(in_features=32768, out_features=4096, bias=False)
        )
        (norm): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
        (norm_context): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
      )
      (1): PreNorm(
        (fn): FeedForward(
          (net): Sequential(
            (0): Linear(in_features=4096, out_features=32768, bias=True)
            (1): GEGLU()
            (2): Linear(in_features=16384, out_features=4096, bias=True)
          )
        )
        (norm): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
      )
    )
  )
  (embedding_model): BidirectionalMistralModel(
    (embed_tokens): Embedding(32000, 4096)
    (la

## 7. Encode helper


In [9]:
def encode_texts(texts, batch_size):
    values = list(texts)
    chunks = []

    for start in range(0, len(values), batch_size):
        batch = values[start:start + batch_size]
        with torch.inference_mode():
            emb = model.encode(batch, instruction="", max_length=MAX_SEQ_LENGTH)
            if TRUNCATE_DIM is not None:
                emb = emb[:, :TRUNCATE_DIM]
            if NORMALIZE_EMBEDDINGS:
                emb = F.normalize(emb, p=2, dim=1)
        chunks.append(emb.detach().cpu().numpy())

    return np.vstack(chunks).astype("float32", copy=False)


## 8. Encode meta


In [10]:
meta = encode_texts(profiles["meta_text"], META_BATCH_SIZE)
meta.shape

C:\Users\User\.cache\huggingface\modules\transformers_modules\3fa59658547db50a1e8e3346cf057fd0c77ed6ef\modeling_nvembed.py:347: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'input_ids': torch.tensor(batch_dict.get('input_ids').to(batch_dict.get('input_ids')).long()),
C:\Users\User\.conda\conda.new\Lib\contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


KeyboardInterrupt: 

## 9. Encode descriptions


In [ ]:
description = encode_texts(profiles["description_text"], DESC_BATCH_SIZE)
description.shape

## 10. Small sanity check


In [ ]:
assert meta.shape == description.shape
assert meta.shape[0] == len(profiles)
assert np.isfinite(meta).all()
assert np.isfinite(description).all()

{
    "items": len(profiles),
    "embedding_dim": meta.shape[1],
    "meta_norm_mean": float(np.linalg.norm(meta, axis=1).mean()),
    "description_norm_mean": float(np.linalg.norm(description, axis=1).mean()),
}

## 11. Save bundle


In [ ]:
np.savez_compressed(
    EMBEDDING_PATH,
    item_idx=profiles["item_idx"].to_numpy("int64"),
    movie_id=profiles["movie_id"].to_numpy("int64"),
    meta=meta,
    description=description,
)

manifest = {
    "run_name": RUN_NAME,
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "source_csv": str(SOURCE_CSV),
    "profile_path": str(PROFILE_PATH),
    "embedding_path": str(EMBEDDING_PATH),
    "model_name": MODEL_NAME,
    "max_seq_length": MAX_SEQ_LENGTH,
    "truncate_dim": TRUNCATE_DIM,
    "normalize_embeddings": NORMALIZE_EMBEDDINGS,
    "n_items": int(len(profiles)),
    "embedding_dim": int(meta.shape[1]),
}

MANIFEST_PATH.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
EMBEDDING_PATH, MANIFEST_PATH

## 12. Reload check


In [ ]:
bundle = np.load(EMBEDDING_PATH)
{k: bundle[k].shape for k in bundle.files}

## 13. Tiny semantic check


In [ ]:
same = np.sum(meta * description, axis=1)
rng = np.random.default_rng(42)
random = np.sum(meta * description[rng.permutation(len(description))], axis=1)

{
    "same_meta_desc_cosine_mean": float(same.mean()),
    "random_meta_desc_cosine_mean": float(random.mean()),
    "gap": float(same.mean() - random.mean()),
}